# sentiment_multiseed — Validazione a 5 seed (source + adattamento)

Ripete l'adattamento a 2 bracci x 3 target già fatto in `sentiment_calibration.ipynb`
(che resta invariato, a singolo seed: training source seed=42, adattamento seed=1), ma su
**5 seed indipendenti** (`SEEDS = [0, 1, 2, 3, 4]`, stessa convenzione dichiarata nei TODO
degli altri notebook del progetto). Qui il seed varia **sia il training del source model
SIA l'adattamento** -- non solo l'adattamento: per ciascun seed si riaddestra `electronics`
da zero (nuovo split train/val/test, nuovo vocabolario TF-IDF, nuovi pesi), si rifitta la
Laplace, si ricontrolla la convergenza MC, e si riesegue l'adattamento a 2 bracci su tutti e
tre i target con quello stesso seed.

**Nessuna duplicazione di codice**: caricamento dati e TF-IDF da `amazon_reviews/data_utils.py`,
modello da `amazon_reviews/model.py` (invariati). Il training del source, che prima viveva
solo dentro `train.py::main()`, è stato estratto in `train.py::train_source_model(seed, ...)`
(usata anche dal vecchio `main()`, comportamento verificato identico -- bit-per-bit lo stesso
checkpoint di prima, confermato da `amazon_reviews/verify_source.py`). Il fit di Laplace, il
controllo di convergenza, la decomposizione BALD e il loop di adattamento a 2 bracci, che
prima vivevano solo dentro le celle di `sentiment_calibration.ipynb`, sono stati estratti nel
nuovo `amazon_reviews/adapt.py` (`fit_laplace_and_check_convergence`, `compute_predictive_results`,
`run_three_arm_adaptation`) -- verificato per riprodurre esattamente gli stessi numeri del
notebook a singolo seed prima di lanciare i 5 seed.

## 1. Stima del tempo totale, prima di lanciare i 5 seed per intero

Basata sui tempi già misurati in `sentiment_calibration.ipynb` (non una nuova misura): il
benchmark preliminare lì aveva trovato **~0.69s/step** (full batch N=2000, `lr=1e-2`), e il
caricamento+training del source **~4.7s**. Un giro completo per seed è: training source
(~4.7s) + fit Laplace/convergenza (non esplicitamente cronometrato in quel notebook, stima
approssimativa ~5s per un K=2/Dp=65 così piccolo) + 3 target x 3 bracci x 100 step
(`ADAPT_STEPS=100`, stessa convenzione) = 900 step x 0.69s = **621s**. Totale per seed
stimato: **~631s (~10.5 minuti)**; per 5 seed: **~3155s (~52.6 minuti)**.

In [ ]:
EST_TRAIN_S = 4.7
EST_LAPLACE_S = 5.0   # non cronometrato esplicitamente in sentiment_calibration.ipynb -- stima approssimativa
EST_STEP_S = 0.6918   # da sentiment_calibration.ipynb, cella di benchmark
N_TARGETS, N_ARMS, ADAPT_STEPS = 3, 2, 100

est_adapt_s = N_TARGETS * N_ARMS * ADAPT_STEPS * EST_STEP_S
est_per_seed_s = EST_TRAIN_S + EST_LAPLACE_S + est_adapt_s
N_SEEDS = 5
est_total_s = est_per_seed_s * N_SEEDS

print(f"stima per seed: training={EST_TRAIN_S}s + laplace/convergenza~{EST_LAPLACE_S}s + "
      f"adattamento({N_TARGETS}x{N_ARMS}x{ADAPT_STEPS} step)={est_adapt_s:.0f}s  "
      f"= {est_per_seed_s:.0f}s (~{est_per_seed_s/60:.1f} min)")
print(f"stima TOTALE per {N_SEEDS} seed: {est_total_s:.0f}s (~{est_total_s/60:.1f} minuti)")
print(f"\n(questa è una stima PRIMA di lanciare la cella lunga sotto -- il tempo osservato "
      f"potrà differire, si veda la Sezione 3 dopo l'esecuzione)")

stima per seed: training=4.7s + laplace/convergenza~5.0s + adattamento(3x2x100 step)=415s  = 425s (~7.1 min)
stima TOTALE per 5 seed: 2124s (~35.4 minuti)

(questa è una stima PRIMA di lanciare la cella lunga sotto -- il tempo osservato potrà differire, si veda la Sezione 3 dopo l'esecuzione)


## 2. Verifica di M_FIXED su più seed, prima di assumerlo fisso

L'istruzione originale permetteva di riusare `M_FIXED` del seed 0 se la convergenza si
stabilizza rapidamente, verificandolo su almeno 2 seed prima di assumerlo per gli altri 3.
Dato che il costo del controllo di convergenza è basso (~9s per seed, dalla stima sopra), lo
**ricalcolo per ciascuno dei 5 seed invece di assumerlo** -- più conservativo del minimo
richiesto, e permette di riportare la sua variabilità reale invece di un'assunzione non
verificata. **Anticipazione del risultato (Sezione 3): `M_FIXED` NON è stabile fra seed**
(2000 per 3 seed su 5, 5000 per gli altri 2) -- ricalcolarlo per ogni seed si è rivelato
necessario, non solo prudente.

## 3. Esecuzione dei 5 seed: training source + Laplace + convergenza + BALD + adattamento a 2 bracci x 3 target

Setup identico a `sentiment_calibration.ipynb` per ciascun seed: `tau_prior = weight_decay *
N_source` (N_source del TRAIN split di quel seed, non delle 2.000 recensioni totali), stesso
seed condiviso fra i 2 bracci e i 3 target all'interno di ciascun run (nessuna variabilità di
seed introdotta come confondimento nel confronto fra bracci o fra target, stessa logica già
usata nel notebook a singolo seed).

In [ ]:
import sys, time
sys.path.insert(0, "amazon_reviews")
sys.path.insert(0, ".")
import numpy as np, torch

from train import train_source_model, to_tensor
from data_utils import load_domain, transform_tfidf
from adapt import extract_features, fit_laplace_and_check_convergence, compute_predictive_results, run_three_arm_adaptation
from code_v2.src.bayesian_model import augment

SEEDS = [0, 1, 2, 3, 4]
TARGET_DOMAINS = ["dvd", "kitchen", "books"]
ADAPT_STEPS = 100

results_per_seed = {}
t_all0 = time.time()
for seed in SEEDS:
    t_seed0 = time.time()
    print(f"\n{'#'*70}\n# SEED {seed}\n{'#'*70}")

    res = train_source_model(seed=seed, verbose=False)
    model, vocab, transformer = res["model"], res["vocab"], res["transformer"]
    counts_source, labels_source = res["counts_source"], res["labels_source"]
    train_idx, test_idx = res["train_idx"], res["test_idx"]
    print(f"  train_source_model: {time.time()-t_seed0:.1f}s  "
          f"source_test_acc={100*res['source_test_acc']:.2f}%  "
          f"target_test_acc={ {k: round(100*v,2) for k,v in res['target_test_acc'].items()} }")

    t_lap0 = time.time()
    X_train_sparse = transform_tfidf([counts_source[i] for i in train_idx], vocab, transformer)
    X_train, y_train = to_tensor(X_train_sparse, labels_source[train_idx])
    Phi_train, _ = extract_features(model, X_train, y_train)
    Phi_aug_train = augment(Phi_train)
    tau_prior = res["weight_decay"] * Phi_train.shape[0]

    X_test_sparse = transform_tfidf([counts_source[i] for i in test_idx], vocab, transformer)
    X_test, y_test = to_tensor(X_test_sparse, labels_source[test_idx])
    Phi_test, y_test_np = extract_features(model, X_test, y_test)

    Phi_aug_eval, y_eval = {"electronics": augment(Phi_test)}, {"electronics": y_test_np}
    X_target_by_domain, y_target_by_domain = {}, {}
    for d in TARGET_DOMAINS:
        counts_t, labels_t = load_domain(d)
        Xt_sparse = transform_tfidf(counts_t, vocab, transformer)
        Xt, yt = to_tensor(Xt_sparse, labels_t)
        Phi_t, y_t_np = extract_features(model, Xt, yt)
        Phi_aug_eval[d] = augment(Phi_t)
        y_eval[d] = y_t_np
        X_target_by_domain[d] = Xt
        y_target_by_domain[d] = yt

    fit_result = fit_laplace_and_check_convergence(model, Phi_aug_train, tau_prior, Phi_aug_eval,
                                                    ["electronics"] + TARGET_DOMAINS, verbose=False)
    laplace, M_FIXED = fit_result["laplace"], fit_result["M_FIXED"]
    laplace_time = time.time() - t_lap0
    print(f"  laplace+convergence: {laplace_time:.1f}s  M_FIXED={M_FIXED}")

    predictive_results = compute_predictive_results(laplace, Phi_aug_eval, y_eval, M_FIXED)
    ratios = {}
    for d in ["electronics"] + TARGET_DOMAINS:
        r = predictive_results[d]
        alea, epi = r["aleatoric"].mean(), r["epistemic"].mean()
        ratios[d] = dict(alea=float(alea), epi=float(epi), ratio=float(alea/epi))
        print(f"    {d}: alea={alea:.4f} epi={epi:.4f} ratio={alea/epi:.2f}x")

    t_adapt0 = time.time()
    adaptation_results = run_three_arm_adaptation(model, laplace, X_target_by_domain, y_target_by_domain,
                                                  adapt_steps=ADAPT_STEPS, seed=seed, verbose=True)
    adapt_time = time.time() - t_adapt0
    print(f"  adaptation (3 target x 2 arm): {adapt_time:.1f}s")

    results_per_seed[seed] = dict(
        source_test_acc=res["source_test_acc"], target_test_acc=res["target_test_acc"],
        M_FIXED=M_FIXED, ratios=ratios,
        adaptation={d: {a: dict(acc_pre=float(v["acc_pre"]), acc_post=float(v["acc_post"]),
                                n_classes_used=v["n_classes_used"])
                        for a, v in arms.items()} for d, arms in adaptation_results.items()},
    )
    print(f"  TOTALE SEED {seed}: {time.time()-t_seed0:.1f}s")

print(f"\nTOTALE COMPLESSIVO: {time.time()-t_all0:.1f}s")

######################################################################
# SEED 0
######################################################################
  train_source_model: 4.9s  source_test_acc=87.00%  target_test_acc={'books': 69.9, 'dvd': 72.8, 'kitchen': 84.85}
  laplace+convergence: 8.5s  M_FIXED=5000
    electronics: alea=0.4127 epi=0.0015 ratio=279.38x
    dvd: alea=0.5202 epi=0.0017 ratio=304.04x
    kitchen: alea=0.4197 epi=0.0014 ratio=292.51x
    books: alea=0.5441 epi=0.0018 ratio=309.59x
dvd/shot_im (seed=0): pre=72.80%  post=73.60%  delta=+0.80pp  classi=2/2
dvd/u_sfan (seed=0): pre=72.80%  post=74.40%  delta=+1.60pp  classi=2/2
kitchen/shot_im (seed=0): pre=84.85%  post=85.70%  delta=+0.85pp  classi=2/2
kitchen/u_sfan (seed=0): pre=84.85%  post=85.25%  delta=+0.40pp  classi=2/2
books/shot_im (seed=0): pre=69.90%  post=72.00%  delta=+2.10pp  classi=2/2
books/u_sfan (seed=0): pre=69.90%  post=73.00%  delta=+3.10pp  classi=2/2
  adaptation (3 target x 2 arm): 12.4s
  TOTALE

**Tempo effettivo: 132.2s (~2.2 minuti) -- molto meno della stima di ~35.4 minuti.** La
discrepanza (~16x) non è un errore nella stima ma nella premessa che la sottendeva: il
benchmark di ~0.69s/step in `sentiment_calibration.ipynb` era una misura di UN SOLO step,
eseguito subito dopo l'avvio del kernel -- probabilmente dominato da costi una tantum (prima
chiamata a `LastLayerLaplace.predictive`, cache BLAS/thread non ancora scaldate, prima
allocazione delle strutture numpy/torch coinvolte), non rappresentativo del costo a regime.
Qui, con centinaia di step consecutivi già dallo stesso seed, il costo a regime è molto più
basso (~0.02s/step effettivo, stimato da adaptation_time/600 step ≈ 12.5s/600). Lezione per
stime future in questo progetto: un benchmark a un solo step sovrastima il costo per step di
oltre un ordine di grandezza quando il collo di bottiglia include inizializzazioni una tantum
-- meglio cronometrare un piccolo blocco di step consecutivi (es. 10) e dividere, non un
singolo step isolato.

## 4. Decomposizione BALD: rapporto aleatoria/epistemica, media ± std su 5 seed

In [ ]:
DOMAINS_BALD = ["electronics", "dvd", "kitchen", "books"]
print(f"{'dominio':>12s} {'media':>10s} {'std':>8s} {'valori (uno per seed)'}")
print("-" * 70)
for d in DOMAINS_BALD:
    vals = [results_per_seed[s]["ratios"][d]["ratio"] for s in SEEDS]
    print(f"{d:>12s} {np.mean(vals):9.2f}x {np.std(vals, ddof=1):7.2f}x   {[round(v,1) for v in vals]}")

print(f"\nM_FIXED per seed: { {s: results_per_seed[s]['M_FIXED'] for s in SEEDS} }")

     dominio      media      std  valori (uno per seed)
----------------------------------------------------------------------
 electronics    177.47x   87.56x   [279.4, 122.9, 117.2, 266.4, 101.4]
         dvd    187.67x   95.66x   [304.0, 126.7, 122.2, 279.3, 106.1]
     kitchen    185.79x   89.08x   [292.5, 130.2, 125.3, 272.8, 108.2]
       books    192.49x   95.82x   [309.6, 130.9, 127.4, 283.7, 110.9]

M_FIXED per seed: {0: 5000, 1: 2000, 2: 2000, 3: 2000, 4: 5000}


**Il rapporto aleatoria/epistemica è esso stesso molto più variabile fra seed di training
che fra domini, a parità di seed.** La std (~88-96x) è quasi grande quanto la media
(~177-192x): il seed 0 dà un rapporto ~280-310x, il seed 4 dà ~101-111x -- un fattore ~3x di
differenza dovuto SOLO al seed di training del source, più ampio della differenza fra domini
a parità di seed (electronics 177x vs. books 192x in media, un +9%). Anche `M_FIXED` non è
stabile (2000 per 3 seed, 5000 per 2) -- confermando quanto anticipato alla Sezione 2:
ricalcolarlo per ogni seed era necessario. Questo è un risultato a sé importante: il singolo
numero "~100x" riportato in `sentiment_calibration.ipynb` (seed=42) era una stima
ragionevole ma non rappresentativa della variabilità reale -- un secondo seed a caso avrebbe
potuto restituire un numero fra 100x e 310x.

## 5. Tre tabelle (una per target): accuracy pre/post/delta, media ± std sui 3 bracci

In [ ]:
ARMS = ["shot_im", "u_sfan"]
TARGETS = ["dvd", "kitchen", "books"]

summary = {}
for t in TARGETS:
    summary[t] = {}
    print(f"=== target: {t} ===")
    print(f"{'braccio':>15s} {'pre':>16s} {'post':>16s} {'delta':>16s}")
    print("-" * 66)
    for a in ARMS:
        pre = np.array([results_per_seed[s]["adaptation"][t][a]["acc_pre"] for s in SEEDS])
        post = np.array([results_per_seed[s]["adaptation"][t][a]["acc_post"] for s in SEEDS])
        delta = post - pre
        summary[t][a] = dict(pre=pre, post=post, delta=delta)
        print(f"{a:>15s} {100*pre.mean():6.2f}%+-{100*pre.std(ddof=1):4.2f} "
              f"{100*post.mean():6.2f}%+-{100*post.std(ddof=1):4.2f} "
              f"{100*delta.mean():+6.2f}pp+-{100*delta.std(ddof=1):4.2f}")
    print()

=== target: dvd ===
        braccio              pre             post            delta
------------------------------------------------------------------
        shot_im  71.58%+-1.28  74.25%+-1.16  +2.67pp+-1.40
         u_sfan  71.58%+-1.28  74.27%+-0.72  +2.69pp+-0.82

=== target: kitchen ===
        braccio              pre             post            delta
------------------------------------------------------------------
        shot_im  84.88%+-0.38  84.90%+-1.16  +0.02pp+-0.93
         u_sfan  84.88%+-0.38  85.40%+-0.79  +0.52pp+-0.52

=== target: books ===
        braccio              pre             post            delta
------------------------------------------------------------------
        shot_im  70.28%+-1.07  72.43%+-0.62  +2.15pp+-0.96
         u_sfan  70.28%+-1.07  73.63%+-0.75  +3.35pp+-0.91



## 6. Wilcoxon signed-rank, accoppiato per seed

L'accoppiamento è legittimo perché stesso seed = stesso source model (stesso training) e
stessa inizializzazione/ordine dei minibatch per entrambi i bracci in quel run -- confronto
appaiato, non fra campioni indipendenti. Un confronto per target: shot_im vs. u_sfan, sui
delta di accuracy (post - pre) di ciascun seed.

In [ ]:
from scipy.stats import wilcoxon

for t in TARGETS:
    d_shot, d_usfan = summary[t]["shot_im"]["delta"], summary[t]["u_sfan"]["delta"]
    stat, p = wilcoxon(d_shot, d_usfan)
    print(f"=== {t} ===")
    print(f"  shot_im vs u_sfan:          W={stat:.1f}  p={p:.4f}")
    print(f"    differenze (pp), una per seed: {[round(100*x,2) for x in (d_shot-d_usfan)]}")
    print()

=== dvd ===
  shot_im vs u_sfan:          W=5.0  p=0.6250
    differenze (pp), una per seed: [-0.8, -0.25, 1.8, -0.3, -0.55]

=== kitchen ===
  shot_im vs u_sfan:          W=3.0  p=0.3125
    differenze (pp), una per seed: [0.45, 0.15, -1.1, -1.4, -0.6]

=== books ===
  shot_im vs u_sfan:          W=0.0  p=0.0625
    differenze (pp), una per seed: [-1.0, -0.95, -1.4, -1.6, -1.05]



**Nessun confronto raggiunge la significatività convenzionale (p<0.05) -- atteso con solo
5 seed** (il p-value minimo possibile per Wilcoxon a due code con n=5 è 0.0625, il floor
combinatorio, non un artefatto di questi dati). Un risultato però si avvicina molto ed è
qualitativamente diverso dagli altri: **su books, u_sfan batte shot_im in tutti e 5 i seed**
(differenze [-1.0, -0.95, -1.4, -1.6, -1.05]pp, stesso segno ovunque, `W=0.0` -- il valore
minimo possibile, `p=0.0625` -- il p-value minimo possibile con n=5). Questo è esattamente il
pattern che ci si aspetterebbe da un effetto reale ma con potenza statistica insufficiente:
con un sesto seed nella stessa direzione, il test diventerebbe significativo. Gli altri due
target (dvd, kitchen) non mostrano questa consistenza di segno: i risultati cambiano
direzione da un seed all'altro, coerente con un margine reale vicino allo zero o rumore di
ottimizzazione dominante -- anche se, notevolmente, la MEDIA punta comunque nella stessa
direzione (u_sfan sopra shot_im) su tutti e tre i target (Sezione 5).

## 7. Braccio vincente: confronto 1-seed (sentiment_calibration.ipynb) vs. media 5-seed

In [ ]:
single_seed_winner = {"dvd": "u_sfan", "kitchen": "u_sfan", "books": "u_sfan"}
print(f"{'target':>10s} {'1-seed (train=42, adapt=1)':>28s} {'media 5-seed':>16s} {'coerente':>10s}")
print("-" * 68)
for t in TARGETS:
    best5 = max(ARMS, key=lambda a: summary[t][a]["delta"].mean())
    coerente = "si" if single_seed_winner[t] == best5 else "NO"
    print(f"{t:>10s} {single_seed_winner[t]:>28s} {best5:>16s} {coerente:>10s}")

    target  1-seed (train=42, adapt=1)     media 5-seed   coerente
--------------------------------------------------------------------
       dvd                       u_sfan           u_sfan         si
   kitchen                       u_sfan           u_sfan         si
     books                       u_sfan           u_sfan         si



**3 su 3 confermati.** Su tutti e tre i target, il braccio vincente osservato con il
singolo seed di `sentiment_calibration.ipynb` (u_sfan, sempre) si conferma anche nella media
a 5 seed. `kitchen` resta il target con il margine più piccolo in assoluto (+0.52pp in
media, contro +2.69pp su dvd e +3.35pp su books) -- coerente con l'essere il target più
vicino al source (shift minimo, meno margine per qualunque intervento di adattamento). Il
Wilcoxon (Sezione 6) non trova un segnale statisticamente distinguibile su dvd/kitchen, ma la
direzione (u_sfan sopra shot_im) è comunque consistente in media su tutti e tre -- un
risultato più pulito, e più facile da interpretare, di quanto la versione con tre bracci
avesse prodotto.

## 8. Collocazione nel confronto multi-esperimento del progetto

Questo è il **primo e unico esperimento del progetto con validazione multi-seed completa**
(source E adattamento, 5 seed indipendenti) -- ogni altro punto della tabella
multi-esperimento (SVHN, MNIST-pieno, e la versione a singolo seed di questo stesso
esperimento) resta a singolo seed, per costruzione o per costo computazionale (i digit
richiedono minuti per seed anche con CNN piccole; qui, MLP su TF-IDF, il costo per 5 seed è
stato di soli 132.2s).

**Che cosa cambia, e che cosa no, per l'interpretazione degli altri esperimenti.** Il fatto
che qui, con validazione multi-seed, il pattern a singolo seed si confermi su tutti e tre i
target (Sezione 7), pur senza raggiungere la significatività statistica formale su due di
essi (Sezione 6), **rafforza retroattivamente la plausibilità, non la certezza**, dei
pattern osservati sui digit e su Rotated-MNIST: mostra che un singolo seed in questa
famiglia di esperimenti (MLP/CNN piccola, IM adaptation full-batch) tende a essere
direzionalmente informativo ma non definitivo, esattamente come i TODO di quei notebook già
avvertivano. Non sostituisce quei risultati con qualcosa di più forte -- li lascia esattamente
al livello di cautela ("verificare con 5+ seed prima di trattarli come risultato di tesi")
che avevano già.

**Un risultato specifico di questo notebook che NON ha equivalente altrove nel progetto:**
la scoperta che il rapporto aleatoria/epistemica del source varia di un fattore ~3x fra seed
di training diversi (Sezione 4, std quasi grande quanto la media) è una osservazione nuova,
non anticipata dai notebook a singolo seed -- suggerisce che qualunque "rapporto
aleatoria/epistemica ~Nx" riportato altrove nel progetto (SVHN ~64x, MNIST-pieno ~12x)
andrebbe interpretato come una stima puntuale da un singolo training, non come una proprietà
stabile del dominio/architettura -- un ulteriore argomento, indipendente da quelli già
discussi nei rispettivi TODO, per una validazione multi-seed anche di quei numeri prima di
trattarli come risultato definitivo.